<a href="https://colab.research.google.com/github/Minhlun25ndtt/bai-tap-python/blob/main/bai-tap-chuong-4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chương 4 — Lớp, tệp và xử lý ngoại lệ (Notebook thực hành)

**Mục tiêu:** lớp/đối tượng, kế thừa, đa hình, `dataclass`; tệp văn bản, JSON, CSV, `pathlib`;
`try/except/else/finally`, `raise` và ngoại lệ tự định nghĩa.

**Phần A** — ví dụ trình diễn cùng giảng viên; **Phần B** — bài tập và thực hành.

## TH 4.1

Viết lớp `HinhCau` có bán kính `ban_kinh`, các phương thức `dien_tich()` ($4\pi r^2$), `the_tich()` ($\frac43\pi r^3$) và `__str__`. Hàm tạo phải báo `ValueError` nếu bán kính không dương. Kiểm tra với $r=2$ (diện tích $16\pi\approx50{,}265$, thể tích $\frac{32\pi}{3}\approx33{,}510$) và $r=-1$.

In [4]:
# TODO: viết lời giải của bạn ở đây
import math

class HinhCau:
    def __init__(self, ban_kinh):
        if not ban_kinh > 0:
            raise ValueError("Bán kính phải dương")
        self.ban_kinh = ban_kinh

    def dien_tich(self):
        return 4 * math.pi * self.ban_kinh ** 2

    def the_tich(self):
        return 4 / 3 * math.pi * self.ban_kinh ** 3

    def __str__(self):
        return f"HinhCau(r={self.ban_kinh})"
c = HinhCau(2)
print(c)
print(f"Diện tích: {c.dien_tich():.3f}")
print(f"Thể tích: {c.the_tich():.3f}")

try:
    HinhCau(-1)
except ValueError as e:
    print("Lỗi:", e)

HinhCau(r=2)
Diện tích: 50.265
Thể tích: 33.510
Lỗi: Bán kính phải dương


In [ ]:
# TODO: viết lời giải của bạn ở đây


## TH 4.3

Viết lớp `TaiKhoan` (chủ tài khoản, số dư ban đầu mặc định $0$ và *không âm*) với `nap(tien)`, `rut(tien)`, danh sách `lich_su` riêng từng tài khoản. Định nghĩa ngoại lệ `SoDuKhongDu` (kế thừa `Exception`) và dùng nó khi rút quá số dư; số tiền không dương thì báo `ValueError`. Kiểm tra bất biến ``số dư $\ge0$'' (Mệnh đề \ref{md:batbien}) với dãy: số dư đầu $1 000 000$, nạp $500 000$, rút $200 000$, rút $2 000 000$.

In [14]:
class SoDuKhongDu(Exception):
    pass


class TaiKhoan:
    def __init__(self, chu_tk, so_du=0):
        if so_du < 0:
            raise ValueError("Số dư ban đầu không được âm")
        self.chu_tk, self.so_du, self.lich_su = chu_tk, so_du, []

    def nap(self, tien):
        if tien <= 0:
            raise ValueError("Số tiền nạp phải dương")
        self.so_du += tien
        self.lich_su.append(f"Nạp {tien}")

    def rut(self, tien):
        if tien <= 0:
            raise ValueError("Số tiền rút phải dương")
        if tien > self.so_du:
            raise SoDuKhongDu(f"Số dư không đủ: cần rút {tien}, chỉ còn {self.so_du}")
        self.so_du -= tien
        self.lich_su.append(f"Rút {tien}")


tk = TaiKhoan("Minh", 1_000_000)
tk.nap(500_000)
tk.rut(200_000)
try:
    tk.rut(2_000_000)
except SoDuKhongDu as e:
    print("Lỗi:", e)

print("Chủ tài khoản:", tk.chu_tk)
print("Số dư:", tk.so_du)
print("Lịch sử:", tk.lich_su)
assert tk.so_du == 1_300_000 >= 0

Lỗi: Số dư không đủ: cần rút 2000000, chỉ còn 1300000
Chủ tài khoản: Minh
Số dư: 1300000
Lịch sử: ['Nạp 500000', 'Rút 200000']


## TH 4.5

Thiết kế lớp cha `HinhHoc` có phương thức `dien_tich()` chỉ báo `NotImplementedError` (buộc lớp con phải cài đặt) và `__str__`; ba lớp con `HinhTron(r)`, `HinhChuNhat(a, b)`, `HinhTamGiac(a, b, c)` (công thức Heron). Với danh sách gồm hình tròn $r=1$, chữ nhật $3\times4$ và tam giác $(3,4,5)$: in các hình theo diện tích tăng dần và tổng diện tích (phải bằng $\pi+18$), nhờ *đa hình*.

In [7]:
import math

class HinhHoc:
    def dien_tich(self):
        raise NotImplementedError("Lớp con phải cài đặt dien_tich()")

    def __str__(self):
        return f"{type(self).__name__}, diện tích = {self.dien_tich():.4f}"


class HinhTron(HinhHoc):
    def __init__(self, r):
        if r <= 0:
            raise ValueError("Bán kính phải dương")
        self.r = r

    def dien_tich(self):
        return math.pi * self.r ** 2


class HinhChuNhat(HinhHoc):
    def __init__(self, a, b):
        if a <= 0 or b <= 0:
            raise ValueError("Các cạnh phải dương")
        self.a, self.b = a, b

    def dien_tich(self):
        return self.a * self.b


class HinhTamGiac(HinhHoc):
    def __init__(self, a, b, c):
        if min(a, b, c) <= 0:
            raise ValueError("Các cạnh phải dương")
        if a + b <= c or a + c <= b or b + c <= a:
            raise ValueError("Ba cạnh không tạo thành tam giác")
        self.a, self.b, self.c = a, b, c

    def dien_tich(self):
        p = (self.a + self.b + self.c) / 2          # nửa chu vi
        return math.sqrt(p * (p - self.a) * (p - self.b) * (p - self.c))   # Heron


ds = [HinhTron(1), HinhChuNhat(3, 4), HinhTamGiac(3, 4, 5)]

print("Theo diện tích tăng dần:")
for h in sorted(ds, key=lambda h: h.dien_tich()):
    print(" ", h)

tong = sum(h.dien_tich() for h in ds)
print(f"Tổng diện tích: {tong:.4f} (π + 18 = {math.pi + 18:.4f})")
assert math.isclose(tong, math.pi + 18)

# Lớp cha không dùng trực tiếp được
try:
    HinhHoc().dien_tich()
except NotImplementedError as e:
    print("NotImplementedError:", e)

# Dữ liệu sai
for f in (lambda: HinhTamGiac(1, 2, 3), lambda: HinhTron(-1), lambda: HinhChuNhat(0, 5)):
    try:
        f()
    except ValueError as e:
        print("ValueError:", e)

Theo diện tích tăng dần:
  HinhTron, diện tích = 3.1416
  HinhTamGiac, diện tích = 6.0000
  HinhChuNhat, diện tích = 12.0000
Tổng diện tích: 21.1416 (π + 18 = 21.1416)
NotImplementedError: Lớp con phải cài đặt dien_tich()
ValueError: Ba cạnh không tạo thành tam giác
ValueError: Bán kính phải dương
ValueError: Các cạnh phải dương


## TH 4.7

Với tệp `data/goldprice.csv` (giá vàng theo tháng, cột `Date` dạng `yyyy-mm`, `Price`), tính giá trung bình của từng năm đủ $12$ tháng, rồi mức tăng (%) so với năm liền trước. In ba năm tăng mạnh nhất và năm giảm mạnh nhất. (Đáp số biết trước: năm $1980$ tăng mạnh nhất, năm $1981$ giảm mạnh nhất.)

In [16]:
# TODO: viết lời giải của bạn ở đây
import csv
from collections import defaultdict

duong_dan = "goldprice.csv"

gia_theo_nam = defaultdict(list)
with open(duong_dan, newline="", encoding="utf-8") as f:
    for dong in csv.DictReader(f):
        nam = int(dong["Date"][:4])
        gia_theo_nam[nam].append(float(dong["Price"]))

# Chỉ giữ năm đủ 12 tháng
tb_nam = {n: sum(g) / 12 for n, g in sorted(gia_theo_nam.items()) if len(g) == 12}
print(f"Số năm đủ 12 tháng: {len(tb_nam)} (từ {min(tb_nam)} đến {max(tb_nam)})")
print("Năm bị bỏ (thiếu tháng):", [n for n, g in gia_theo_nam.items() if len(g) != 12])

# Mức tăng (%) so với năm liền trước (chỉ tính khi năm trước cũng có dữ liệu)
tang = {n: (tb_nam[n] / tb_nam[n - 1] - 1) * 100 for n in tb_nam if n - 1 in tb_nam}

xep_hang = sorted(tang.items(), key=lambda kv: kv[1], reverse=True)

print("\nBa năm tăng mạnh nhất:")
for n, pt in xep_hang[:3]:
    print(f"  {n}: {pt:+.2f}%  (TB {tb_nam[n - 1]:.2f} → {tb_nam[n]:.2f})")

n, pt = xep_hang[-1]
print("\nNăm giảm mạnh nhất:")
print(f"  {n}: {pt:+.2f}%  (TB {tb_nam[n - 1]:.2f} → {tb_nam[n]:.2f})")

assert xep_hang[0][0] == 1980 and xep_hang[-1][0] == 1981

Số năm đủ 12 tháng: 70 (từ 1950 đến 2019)
Năm bị bỏ (thiếu tháng): [2020]

Ba năm tăng mạnh nhất:
  1980: +99.46%  (TB 307.30 → 612.95)
  1973: +67.15%  (TB 58.10 → 97.12)
  1974: +63.81%  (TB 97.12 → 159.10)

Năm giảm mạnh nhất:
  1981: -24.95%  (TB 612.95 → 460.02)


## TH 4.9 ∗

Tạo tệp `luong.csv` với nội dung `nam_kn,luong` / `1,30` / `2,35` / `abc,40` / `3,` / `4,50` / `5,55` (mỗi gạch chéo là một dòng). Viết chương trình đọc tệp bằng `csv.DictReader`: dòng nào không đổi được sang số thì bị bỏ qua và *ghi lại số thứ tự dòng trong tệp* (dòng tiêu đề là dòng $1$). Dùng ngoại lệ tự định nghĩa `DongKhongHopLe`. In các dòng hợp lệ, danh sách dòng lỗi và lương trung bình của các dòng hợp lệ.

In [17]:
# TODO: viết lời giải của bạn ở đây
import csv

# 1) Tạo tệp luong.csv
with open("luong.csv", "w", newline="", encoding="utf-8") as f:
    f.write("nam_kn,luong\n1,30\n2,35\nabc,40\n3,\n4,50\n5,55\n")


class DongKhongHopLe(Exception):
    def __init__(self, so_dong, noi_dung):
        super().__init__(f"Dòng {so_dong} không hợp lệ: {noi_dung}")
        self.so_dong = so_dong


def doc_dong(dong, so_dong):
    """Đổi một dòng sang (năm kinh nghiệm, lương); nếu không được thì báo DongKhongHopLe."""
    try:
        return int(dong["nam_kn"]), float(dong["luong"])
    except (ValueError, TypeError):
        raise DongKhongHopLe(so_dong, dict(dong))


hop_le, dong_loi = [], []
with open("luong.csv", newline="", encoding="utf-8") as f:
    doc = csv.DictReader(f)
    for dong in doc:
        try:
            hop_le.append(doc_dong(dong, doc.line_num))   # line_num: dòng tiêu đề là 1
        except DongKhongHopLe as e:
            dong_loi.append(e.so_dong)
            print("Bỏ qua:", e)

print("\nCác dòng hợp lệ:")
for nam, luong in hop_le:
    print(f"  nam_kn = {nam}, luong = {luong:g}")
print("Các dòng lỗi (số thứ tự trong tệp):", dong_loi)

if hop_le:
    tb = sum(l for _, l in hop_le) / len(hop_le)
    print(f"Lương trung bình ({len(hop_le)} dòng hợp lệ): {tb:g}")
else:
    print("Không có dòng hợp lệ nào")

assert dong_loi == [4, 5] and tb == 42.5

Bỏ qua: Dòng 4 không hợp lệ: {'nam_kn': 'abc', 'luong': '40'}
Bỏ qua: Dòng 5 không hợp lệ: {'nam_kn': '3', 'luong': ''}

Các dòng hợp lệ:
  nam_kn = 1, luong = 30
  nam_kn = 2, luong = 35
  nam_kn = 4, luong = 50
  nam_kn = 5, luong = 55
Các dòng lỗi (số thứ tự trong tệp): [4, 5]
Lương trung bình (4 dòng hợp lệ): 42.5
